In [1]:
# installs

In [2]:
# imports

import os
import shutil
from pathlib import Path
import pandas as pd
import cv2
import matplotlib.pyplot as 

In [3]:
target = Path("data/metadata.csv")

if not target.exists(): 
    def create_metadata():
        celeb_real = Path("raw_datasets/Celeb-real")
        celeb_fake = Path("raw_datasets/Celeb-synthesis")
        ff_real = Path("raw_datasets/FaceForensics++_C23/original")
        ff_fake = Path("raw_datasets/FaceForensics++_C23")
        ff_fake_dirs = ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
    
        data = []
        
        base_dir = Path("data")
    
        for vid in celeb_real.glob("*.mp4"):
            data.append({
                "path": str(vid),
                "source": "Celeb-DF",
                "label": 0
            })
    
        for vid in celeb_fake.glob("*.mp4"):
            data.append({
                "path": str(vid),
                "source": "Celeb-DF",
                "label": 1
            })
    
        for vid in ff_real.glob("*.mp4"):
            data.append({
                "path": str(vid),
                "source": "FF++",
                "label": 0
            })
    
        for dirs in ff_fake_dirs:
            for vid in (ff_fake/dirs).glob("*.mp4"):
                data.append({
                "path": str(vid),
                "source": "FF++",
                "label": 1
            })
    
        df = pd.DataFrame(data);
        Path("data").mkdir(exist_ok=True)
        df.to_csv(target, index=False)
    
        print("**************************")
        print("Metadata Created Successfully")
        print("**************************")
    
    
    create_metadata()

In [4]:
df = pd.read_csv("data/metadata.csv")
df

,path,source,label
0,raw_datasets\Celeb-real\id0_0000.mp4,Celeb-DF,0
1,raw_datasets\Celeb-real\id0_0001.mp4,Celeb-DF,0
2,raw_datasets\Celeb-real\id0_0002.mp4,Celeb-DF,0
3,raw_datasets\Celeb-real\id0_0003.mp4,Celeb-DF,0
4,raw_datasets\Celeb-real\id0_0004.mp4,Celeb-DF,0
...,...,...,...
12224,raw_datasets\FaceForensics++_C23\NeuralTexture...,FF++,1
12225,raw_datasets\FaceForensics++_C23\NeuralTexture...,FF++,1
12226,raw_datasets\FaceForensics++_C23\NeuralTexture...,FF++,1
12227,raw_datasets\FaceForensics++_C23\NeuralTexture...,FF++,1


In [5]:
df = df.sample(frac=1, random_state=42)
train_size = int(0.7 * len(df))
train_df = df[:train_size]
val_test_size = int(0.15 * len(df))
val_df = df[train_size: train_size + val_test_size]
test_df = df[train_size + val_test_size:]

print("Train data:", len(train_df))
print("Val data:", len(val_df))
print("Test data:", len(test_df))

Train data: 8560
Val data: 1834
Test data: 1835


In [8]:
def extract_frames(vid_path, num_frames = 30, resize=None):
    vid_path = Path(vid_path)

    cap = cv2.VideoCapture(str(vid_path))

    if not cap.isOpened():
        print(f"Error opening video: {vid_path}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        print(f"No frames found in: {vid_path}")
        return

    interval = max(total_frames // num_frames, 1)

    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % interval == 0:

            if resize is not None:
                frame = cv2.resize(frame, resize)

            frames.append(frame)

            if len(frames) >=  num_frames:
                break

        frame_id += 1

    cap.release()
    return frames


KeyboardInterrupt: 